In [3]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

# 1. Load Dataset and Handle Missing Values
df = pd.read_csv("creditcard.csv")

# Clean missing values before splitting
df = df.dropna()

# Features 'Time' and 'Amount' need scaling
scaler = StandardScaler()
df["NormalizedAmount"] = scaler.fit_transform(df["Amount"].values.reshape(-1, 1))
df = df.drop(["Time", "Amount"], axis=1)

X = df.drop("Class", axis=1)
y = df["Class"]

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


# 2. Handle Class Imbalance using SMOTE

print("\nApplying SMOTE to training data...")
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
print("Resampled shape:", X_train_resampled.shape)


# 3. Train Models (Logistic Regression & XGBoost)

print("\nTraining Logistic Regression...")
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_resampled, y_train_resampled)

print("Training XGBoost...")
xgb_model = XGBClassifier(
    n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, n_jobs=-1
)
xgb_model.fit(X_train_resampled, y_train_resampled)

# 4. Evaluate Using ROC-AUC and Confusion Matrix

models = {"Logistic Regression": lr_model, "XGBoost": xgb_model}

for name, model in models.items():
    print(f"\n--- Evaluation for {name} ---")
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print("ROC-AUC Score:", roc_auc_score(y_test, y_prob))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

# 5. Simulate Real-Time Detection
print("\nSimulating Real-Time Detection with a subset of test data...")
# Take a random sample of 5 transactions from the test set to simulate incoming live traffic
simulation_subset = X_test.sample(n=5, random_state=42)
simulation_actuals = y_test.loc[simulation_subset.index]

# Predict using the XGBoost model
simulated_preds = xgb_model.predict(simulation_subset)
simulated_probs = xgb_model.predict_proba(simulation_subset)[:, 1]

for i, idx in enumerate(simulation_subset.index):
    status = (
        " FRAUD DETECTED!"
        if simulated_preds[i] == 1
        else "Legitimate Transaction"
    )
    print(
        f"Transaction ID {idx} -> Prediction: {status} (Fraud Probability: {simulated_probs[i]:.4f})"
    )


Applying SMOTE to training data...
Resampled shape: (12714, 29)

Training Logistic Regression...
Training XGBoost...

--- Evaluation for Logistic Regression ---
ROC-AUC Score: 1.0

Confusion Matrix:
[[1590    0]
 [   0    5]]

Classification Report:
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00      1590
         1.0       1.00      1.00      1.00         5

    accuracy                           1.00      1595
   macro avg       1.00      1.00      1.00      1595
weighted avg       1.00      1.00      1.00      1595


--- Evaluation for XGBoost ---
ROC-AUC Score: 1.0

Confusion Matrix:
[[1590    0]
 [   0    5]]

Classification Report:
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00      1590
         1.0       1.00      1.00      1.00         5

    accuracy                           1.00      1595
   macro avg       1.00      1.00      1.00      1595
weighted avg       1.00      